# Predykcja cen samochodów – Model bazowy (Baseline)

**Problem:** Regresja – przewidywanie ceny samochodu na podstawie jego cech.

**Dataset:** Car Price Prediction (dane z 2021 roku)

Notebook zawiera:
1. Eksplorację danych (EDA)
2. Preprocessing
3. Trening modelu bazowego
4. Ewaluację


## 0. Importy i konfiguracja


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

RANDOM_STATE = 42
DATA_PATH = '../data/01_raw/train.csv'

print('Importy załadowane pomyślnie.')


Importy załadowane pomyślnie.


## 1. Eksploracja danych (EDA)


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Kształt danych: {df.shape}')
df.head()


In [ ]:
print('Typy danych:')
print(df.dtypes)
print(f'\nBrakujące wartości:')
print(df.isnull().sum())


In [ ]:
df.describe()


In [ ]:
# Rozkład ceny (zmienna docelowa)
# Usunięcie wierszy z niepoprawnymi cenami
df_clean = df[df['Price'] != '-'].copy()
df_clean['Price'] = pd.to_numeric(df_clean['Price'], errors='coerce')
df_clean = df_clean.dropna(subset=['Price'])
df_clean = df_clean[df_clean['Price'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['Price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Rozkład ceny (oryginalna skala)')
axes[0].set_xlabel('Cena')
axes[0].set_ylabel('Liczba samochodów')

axes[1].hist(np.log1p(df_clean['Price']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Rozkład ceny (skala logarytmiczna)')
axes[1].set_xlabel('log(Cena + 1)')
axes[1].set_ylabel('Liczba samochodów')

plt.tight_layout()
plt.savefig('../data/08_reporting/price_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Liczba rekordów po usunięciu błędnych cen: {len(df_clean)}')


In [ ]:
# Top producenci
top_manufacturers = df_clean['Manufacturer'].value_counts().head(15)
plt.figure(figsize=(12, 5))
top_manufacturers.plot(kind='bar', color='steelblue')
plt.title('Top 15 producentów samochodów')
plt.xlabel('Producent')
plt.ylabel('Liczba ogłoszeń')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Cena wg kategorii
plt.figure(figsize=(12, 5))
df_clean.groupby('Category')['Price'].median().sort_values(ascending=False).plot(kind='bar', color='coral')
plt.title('Mediana ceny wg kategorii samochodu')
plt.xlabel('Kategoria')
plt.ylabel('Mediana ceny')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Rok produkcji vs cena
plt.figure(figsize=(12, 5))
df_clean.groupby('Prod. year')['Price'].median().plot(color='green', marker='o', markersize=3)
plt.title('Mediana ceny wg roku produkcji')
plt.xlabel('Rok produkcji')
plt.ylabel('Mediana ceny')
plt.tight_layout()
plt.show()


In [ ]:
# Rozkład typów paliwa
fuel_counts = df_clean['Fuel type'].value_counts()
plt.figure(figsize=(8, 5))
fuel_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Rozkład typów paliwa')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 2. Preprocessing


In [ ]:
def preprocess(df):
    df = df.copy()

    # Usunięcie kolumny ID
    df = df.drop(columns=['ID'], errors='ignore')

    # Czyszczenie Price
    df = df[df['Price'] != '-']
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df = df.dropna(subset=['Price'])
    df = df[df['Price'] > 0]

    # Czyszczenie Levy
    df['Levy'] = df['Levy'].replace('-', np.nan)
    df['Levy'] = pd.to_numeric(df['Levy'], errors='coerce')
    df['Levy'] = df['Levy'].fillna(df['Levy'].median())

    # Czyszczenie Mileage (usunięcie ' km')
    df['Mileage'] = df['Mileage'].str.replace(' km', '', regex=False)
    df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')
    df['Mileage'] = df['Mileage'].fillna(df['Mileage'].median())

    # Czyszczenie Engine volume (usunięcie 'Turbo')
    df['Engine volume'] = df['Engine volume'].str.replace(' Turbo', '', regex=False)
    df['Engine volume'] = pd.to_numeric(df['Engine volume'], errors='coerce')
    df['Engine volume'] = df['Engine volume'].fillna(df['Engine volume'].median())

    # Czyszczenie Cylinders
    df['Cylinders'] = pd.to_numeric(df['Cylinders'], errors='coerce')
    df['Cylinders'] = df['Cylinders'].fillna(df['Cylinders'].median())

    # Wiek samochodu (rok referencyjny 2021)
    df['Car age'] = 2021 - df['Prod. year']
    df = df.drop(columns=['Prod. year'])

    # Czyszczenie Doors
    df['Doors'] = df['Doors'].replace({'04-May': '4', '02-Mar': '2', '>5': '5'})
    df['Doors'] = pd.to_numeric(df['Doors'], errors='coerce')
    df['Doors'] = df['Doors'].fillna(4)

    # Leather interior -> binary
    df['Leather interior'] = (df['Leather interior'] == 'Yes').astype(int)

    # Enkodowanie zmiennych kategorycznych
    cat_cols = ['Manufacturer', 'Model', 'Category', 'Fuel type',
                'Gear box type', 'Drive wheels', 'Wheel', 'Color']

    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    return df, encoders

df_raw = pd.read_csv(DATA_PATH)
df_processed, encoders = preprocess(df_raw)
print(f'Kształt po preprocessingu: {df_processed.shape}')
df_processed.head()


In [ ]:
# Macierz korelacji
numeric_df = df_processed.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Macierz korelacji cech')
plt.tight_layout()
plt.show()


In [ ]:
# Podział na cechy i target
TARGET = 'Price'
FEATURES = [c for c in df_processed.columns if c != TARGET]

X = df_processed[FEATURES]
y = np.log1p(df_processed[TARGET])  # transformacja logarytmiczna

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Zbiór treningowy: {X_train.shape}')
print(f'Zbiór testowy:    {X_test.shape}')


## 3. Trening modelu bazowego


In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    # Metryki w skali logarytmicznej
    mae_log = mean_absolute_error(y_te, y_pred)
    rmse_log = np.sqrt(mean_squared_error(y_te, y_pred))
    r2 = r2_score(y_te, y_pred)

    # Metryki w oryginalnej skali
    y_pred_orig = np.expm1(y_pred)
    y_te_orig = np.expm1(y_te)
    mae_orig = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse_orig = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))

    print(f'\n=== {name} ===')
    print(f'  R²:        {r2:.4f}')
    print(f'  MAE (log): {mae_log:.4f}')
    print(f'  RMSE(log): {rmse_log:.4f}')
    print(f'  MAE (USD): {mae_orig:,.0f}')
    print(f'  RMSE(USD): {rmse_orig:,.0f}')

    return model, {'r2': r2, 'mae_log': mae_log, 'rmse_log': rmse_log,
                   'mae_usd': mae_orig, 'rmse_usd': rmse_orig}

results = {}


In [ ]:
# Model 1: Regresja liniowa (baseline)
lr_model, lr_metrics = evaluate(
    'Regresja liniowa (baseline)',
    LinearRegression(),
    X_train, y_train, X_test, y_test
)
results['Linear Regression'] = lr_metrics


In [ ]:
# Model 2: Random Forest
rf_model, rf_metrics = evaluate(
    'Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train, X_test, y_test
)
results['Random Forest'] = rf_metrics


In [ ]:
# Model 3: Gradient Boosting
gb_model, gb_metrics = evaluate(
    'Gradient Boosting',
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=4, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test
)
results['Gradient Boosting'] = gb_metrics


## 4. Ewaluacja


In [ ]:
# Porównanie modeli
results_df = pd.DataFrame(results).T
print('\n=== Porównanie modeli ===')
print(results_df.to_string())


In [ ]:
# Wykres porównania metryk
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['r2', 'mae_log', 'rmse_log']
titles = ['R² (wyższy = lepszy)', 'MAE (log, niższy = lepszy)', 'RMSE (log, niższy = lepszy)']
colors = ['steelblue', 'coral', 'green']

for ax, metric, title, color in zip(axes, metrics_to_plot, titles, colors):
    vals = results_df[metric]
    ax.bar(vals.index, vals.values, color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xticklabels(vals.index, rotation=15, ha='right')
    for i, v in enumerate(vals.values):
        ax.text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Porównanie modeli ML', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/08_reporting/model_comparison_baseline.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Wykres: Predykcje vs Rzeczywiste (najlepszy model - Gradient Boosting)
y_pred_best = gb_model.predict(X_test)

y_pred_orig = np.expm1(y_pred_best)
y_test_orig = np.expm1(y_test)

# Ograniczenie do rozsądnego zakresu cen
mask = (y_test_orig < 100000) & (y_pred_orig < 100000)

plt.figure(figsize=(8, 8))
plt.scatter(y_test_orig[mask], y_pred_orig[mask], alpha=0.3, s=10, color='steelblue')
max_val = max(y_test_orig[mask].max(), y_pred_orig[mask].max())
plt.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Idealna predykcja')
plt.xlabel('Rzeczywista cena (USD)')
plt.ylabel('Przewidywana cena (USD)')
plt.title('Gradient Boosting: Predykcje vs Rzeczywiste')
plt.legend()
plt.tight_layout()
plt.savefig('../data/08_reporting/predictions_vs_actual.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Ważność cech (Random Forest)
feature_importance = pd.Series(
    rf_model.feature_importances_, index=FEATURES
).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='barh', color='steelblue')
plt.title('Top 15 najważniejszych cech (Random Forest)')
plt.xlabel('Ważność cechy')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../data/08_reporting/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Podsumowanie
best_name = results_df['r2'].idxmax()
best_r2 = results_df.loc[best_name, 'r2']
best_mae = results_df.loc[best_name, 'mae_usd']

print('=' * 50)
print('PODSUMOWANIE')
print('=' * 50)
print(f'Najlepszy model: {best_name}')
print(f'R²:              {best_r2:.4f}')
print(f'MAE (USD):       {best_mae:,.0f}')
print()
print('Wnioski:')
print('- Gradient Boosting osiąga najlepsze wyniki spośród testowanych modeli')
print('- Najważniejsze cechy to: rok produkcji (wiek), przebieg, pojemność silnika')
print('- Transformacja logarytmiczna ceny poprawia jakość modelu')
print('- Model bazowy stanowi punkt wyjścia do dalszej optymalizacji w pipeline Kedro')
